# 12.2 Scaled Dot-Product Attention과 Self-Attention — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter12_2_scaled_dot_attention.ipynb)

책 본문: [12.2 Scaled Dot-Product Attention과 Self-Attention](https://smhanlab.com/book-ml/kor/ml1/chapter12/2.html)

이 노트북은 본문의 scaled dot-product attention을 `numpy`로 구현하고,
네 단어 대명사 예의 어텐션 가중치를 **시각화**한 뒤, \(\sqrt{d_k}\) 스케일링이
**왜** 필요한지(내적의 분산이 차원에 비례) 숫자로 확인합니다.


In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
print("numpy", np.__version__)

numpy 2.4.6


## 1. 어텐션 함수 (본문 코드 그대로)

본문의 `softmax`와 `attention`을 `numpy` 행렬 연산으로 다시 씁니다.
`attention`이 하는 4단계 — \(QK^T\) → \(\sqrt{d_k}\) 스케일 → softmax →
\(V\) 가중합 — 를 한 줄씩 옮긴 것입니다.

In [2]:
def softmax(X, axis=-1):
    # softmax: axis 방향으로 합 1인 분포 (1-D 행 or 2-D 행렬의 각 행 모두 동작)
    m = X.max(axis=axis, keepdims=True)
    e = np.exp(X - m)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, d_k):
    scores = Q @ K.T / np.sqrt(d_k)     # (n x n) 원점수, sqrt(d_k) 스케일
    weights = softmax(scores)           # (n x n) 각 행 합 1
    output = weights @ V                # (n x d_v) 가중합
    return output, weights, scores
print("attention 함수 정의 완료 (scores도 함께 반환)")

attention 함수 정의 완료 (scores도 함께 반환)


## 2. 대명사 지시 관계: [동물, 도로, 건너지, 그것은]

본문 실습과 동일한 4개 임베딩을 \(Q=K=V\)로 self-attention을 계산합니다.
"그것은"이 "동물"과 가깝게 설계되어 있어, "그것은"의 가중치가 "동물" 열에
집중하는 것이 핵심 결과입니다.

In [3]:
words = ["동물", "도로", "건너지", "그것은"]
E = np.array([[3,0,0], [0,3,0], [0,0,3], [2.8,0.3,0]], float)  # d_k = 3
Q = K = V = E
output, weights, scores = attention(Q, K, V, d_k=3)

print("어텐션 가중치 (행 = Query 단어, 열 = Key 단어):")
print("      " + "  ".join(f"{w:>7}" for w in words))
for i, w in enumerate(words):
    print(f"{w:5} " + "  ".join(f"{v:7.3f}" for v in weights[i]))
print("\n'그것은' 행이 '동물' 열에 주는 가중치 =", round(weights[3][0], 3))
print("출력(4x3)의 각 행 = V 원행들의 가중평균(볼록 결합):")
print(np.round(output, 3))

어텐션 가중치 (행 = Query 단어, 열 = Key 단어):
           동물       도로      건너지      그것은
동물      0.582    0.003    0.003    0.412
도로      0.005    0.980    0.005    0.009
건너지     0.005    0.005    0.984    0.005
그것은     0.561    0.007    0.004    0.427

'그것은' 행이 '동물' 열에 주는 가중치 = 0.561
출력(4x3)의 각 행 = V 원행들의 가중평균(볼록 결합):
[[2.898 0.133 0.01 ]
 [0.042 2.943 0.016]
 [0.032 0.018 2.951]
 [2.879 0.15  0.013]]


## 3. 어텐션 가중치 행렬 시각화

`weights` 행렬을 히트맵으로 그립니다. **행 = Query(누가 보나)**,
**열 = Key(누구를 보나)**. 대각선이 밝은 것은 자기 주입(자기 자신과의
내적 \(\|q_i\|^2\)이 최대)이고, "그것은" 행의 "동물" 칸이 밝은 것이
대명사-선행사 관계입니다. 이 SVG가 본문 `ch12_2_attention_heatmap.svg`입니다.

In [4]:
fig, ax = plt.subplots(figsize=(5.6, 4.8))
im = ax.imshow(weights, cmap="viridis")
ax.set_xticks(range(4)); ax.set_xticklabels(words, fontsize=10)
ax.set_yticks(range(4)); ax.set_yticklabels(words, fontsize=10)
ax.set_xlabel("Key (target of attention)", fontsize=10)
ax.set_ylabel("Query (word giving attention)", fontsize=10)
ax.set_title("self-attention weights (Q=K=V, d_k=3)", fontsize=11)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{weights[i][j]:.2f}", ha="center", va="center",
                color="white" if weights[i][j] > 0.25 else "black", fontsize=9)
fig.colorbar(im, ax=ax, label="weight")
fig.tight_layout()
fig.savefig(f"{IMG}/ch12_2_attention_heatmap.svg")
plt.show()
print("저장됨:", f"{IMG}/ch12_2_attention_heatmap.svg")

저장됨: /home/smhan/book-ml/kor/src/images/ch12_2_attention_heatmap.svg


## 4. 원점수 "차이"가 softmax를 어떻게 극단화하는가

2개 원점수 \([0, g]\)에 softmax를 적용하면 두 번째 가중치가 정확히
시그모이드 \(\sigma(g)\)입니다. \(g\)가 커질수록 one-hot에 붙습니다.
(이게 "softmax가 차이를 지수적으로 증폭한다"는 주장의 최소 예.)

In [5]:
import numpy as np
g_vals = [0.5, 2.0, 8.0]
print("  g   softmax([0,g])          시그모이드 σ(g)")
for g in g_vals:
    w = softmax(np.array([0.0, g]))
    sig = 1/(1+np.exp(-g))
    print(f"{g:3}  ({w[0]:.4f}, {w[1]:.4f})      {sig:.5f}   "
          f"일치={np.isclose(w[1], sig)}")

# g를 sweep하며 σ(g) 곡선
g = np.linspace(-4, 4, 200)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(g, 1/(1+np.exp(-g)), color="#d95f02", lw=2, label=r"softmax([0,g])$2$nd weight $=\sigma(g)$")
for g0 in g_vals:
    ax.axvline(g0, color="gray", ls=":", lw=0.8)
    ax.plot([g0],[1/(1+np.exp(-g0))],"o",color="k",ms=5)
ax.set_xlabel(r"score gap $g$"); ax.set_ylabel("weight of the second word")
ax.set_title("softmax = sigmoid(score gap)"); ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(f"{IMG}/ch12_2_softmax_sigmoid.svg")
plt.show()

  g   softmax([0,g])          시그모이드 σ(g)
0.5  (0.3775, 0.6225)      0.62246   일치=True
2.0  (0.1192, 0.8808)      0.88080   일치=True
8.0  (0.0003, 0.9997)      0.99966   일치=True


## 5. \(\sqrt{d_k}\) 스케일링: 1 vs \(1/\sqrt{d_k}\) vs \(1/d_k\)

*같은* 원점수(표준편차 \(\sqrt{64}=8\)인 64개 값)에 세 가지 스케일을
적용합니다. 스케일링 없이 넣으면 one-hot(마지막 가중치 1), \(1/\sqrt{d_k}\)
으로 나누면 분화된 가중치, \(1/d_k\)로 나누면 균등에 붙습니다.

In [6]:
rng = np.random.default_rng(3)
s = rng.normal(0, 1, 64) * np.sqrt(64)   # 원점수: std = 8 (스케일링 안 한 내적)
for name, div in [("스케일링 없음 (1)", 1.0),
                  ("1/sqrt(d_k)", np.sqrt(64)),
                  ("1/d_k", 64.0)]:
    w = softmax(s / div)
    print(f"{name:16s}: max-가중치={w.max():.4f}  top2 합={np.sort(w)[-2:].sum():.4f}  (균등=1/64={1/64:.4f})")

스케일링 없음 (1)     : max-가중치=0.9999  top2 합=1.0000  (균등=1/64=0.0156)
1/sqrt(d_k)     : max-가중치=0.2400  top2 합=0.3065  (균등=1/64=0.0156)
1/d_k           : max-가중치=0.0236  top2 합=0.0437  (균등=1/64=0.0156)


## 6. 차원이 커질수록 스케일링이 왜 필요한가

Query 1개·Key \(d\)개(모두 \(\mathcal{N}(0,1)\))를 무작위 생성하고,
스케일링 *전/후* softmax의 **최대 가중치**를 500회 평균합니다.
\(d\)가 4→256으로 커지면 스케일링 없이 넣은 softmax는 점점 one-hot에
붙지만, \(1/\sqrt{d_k}\) 스케일링은 \(d\)와 무관하게 낮은(분화된)
수준을 유지합니다.

In [7]:
dims = [4, 16, 64, 256]
uns_scaled, sca_scaled = [], []
for d in dims:
    u, sc = [], []
    for _ in range(500):
        Q = rng.normal(0,1,(1,d)); Ks = rng.normal(0,1,(d,d))
        s = Ks @ Q[0]
        u.append(softmax(s).max()); sc.append(softmax(s/np.sqrt(d)).max())
    uns_scaled.append(np.median(u)); sca_scaled.append(np.median(sc))
    print(f"d={d:4d}: 스케일링 전 median max-wt={np.median(u):.4f}  |  1/sqrt(d) 후 median max-wt={np.median(sc):.4f}")

fig, ax = plt.subplots(figsize=(6,3.8))
x = np.arange(len(dims))
ax.plot(x, uns_scaled, "o-", color="#a50f15", lw=2, label="no scaling")
ax.plot(x, sca_scaled, "s-", color="#1a9641", lw=2, label=r"$1/\sqrt{d_k}$ scaling")
ax.set_xticks(x); ax.set_xticklabels([f"d={d}" for d in dims])
ax.set_ylabel("max softmax weight (median)")
ax.set_title(r"Without scaling, softmax collapses to one-hot as $d_k$ grows")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{IMG}/ch12_2_scaling_vs_dim.svg")
plt.show()

d=   4: 스케일링 전 median max-wt=0.6278  |  1/sqrt(d) 후 median max-wt=0.4626
d=  16: 스케일링 전 median max-wt=0.7047  |  1/sqrt(d) 후 median max-wt=0.2186
d=  64: 스케일링 전 median max-wt=0.8720  |  1/sqrt(d) 후 median max-wt=0.0987


d= 256: 스케일링 전 median max-wt=0.9644  |  1/sqrt(d) 후 median max-wt=0.0371


## 7. 볼록성 확인: 출력은 V 원행들의 볼록 껍질 안에 있다

Section 2의 `output`(4×3)이 V(=E)의 4개 원행들의 볼록 결합인지 확인합니다.
각 출력 행이 (a) V 원행들의 *범위* 안에 있는지(필요 조건), (b) 실제로
4개 원행의 *볼록 조합*으로 재구성되는지(가중치가 합 1·비음수이므로
성립해야 함)를 확인합니다.

In [8]:
# (a) 범위(_bbox) 체크 — 볼록 껍질 안에 있으면 반드시 만족
lo, hi = E.min(axis=0), E.max(axis=0)
in_bbox = np.all((output >= lo - 1e-9) & (output <= hi + 1e-9), axis=1)
print("V 원행 범위: min =", lo, " max =", hi)
for w, o in zip(words, output):
    print(f"  {w}: out={np.round(o,3)}  범위 안={bool(in_bbox[words.index(w)])}")

# (b) 가중합 재구성: output[i] == sum_j weights[i][j] * E[j] (정의상 성립)
recon = weights @ E
print("\n볼록 결합 재구성 오차(최대 절댓값) =",
      f"{np.abs(recon - output).max():.2e}  (0에 가까우면 정의와 일치)")
print("=> 각 출력은 V 원행들의 볼록 결합(볼록 껍질 내부)이다.")

V 원행 범위: min = [0. 0. 0.]  max = [3. 3. 3.]
  동물: out=[2.898 0.133 0.01 ]  범위 안=True
  도로: out=[0.042 2.943 0.016]  범위 안=True
  건너지: out=[0.032 0.018 2.951]  범위 안=True
  그것은: out=[2.879 0.15  0.013]  범위 안=True

볼록 결합 재구성 오차(최대 절댓값) = 0.00e+00  (0에 가까우면 정의와 일치)
=> 각 출력은 V 원행들의 볼록 결합(볼록 껍질 내부)이다.


## 8. 정리

| 실험 | 관측 | 본문의 어떤 주장을 확인하나 |
|---|---|---|
| 4단어 self-attention | "그것은"→"동물" 0.561, 자기 0.427 | 대명사-선행사 관계 = 어텐션 가중치 |
| softmax([0,g]) | = 시그모이드 σ(g) | softmax가 원점수 *차이*를 증폭 |
| 1 vs 1/√d vs 1/d (d=64) | 0.9999 / 0.24 / 0.024 | √d_k가 "정규" 스케일 |
| d=4→256 스케일링 전/후 | 0.64→0.95 / 0.46→0.038 | 차원 ↑ → 스케일링 없이 one-hot |
| 출력 = 가중합 | 볼록 결합 재구성 오차 ≈ 0 | 출력은 V 원행들의 볼록 껍질 안에 |